<a href="https://colab.research.google.com/github/EunjeLee0812/Sanhak/blob/seowonryeol/code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# 라이브러리 설치
!pip install faster-whisper rapidfuzz g2pk konlpy python-mecab-ko


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [7]:
import gc, sys
import os, re, json, glob, csv, random, glob, time
from dataclasses import dataclass
from typing import Dict, List, Optional, Any, Tuple
from g2pk import G2p
from faster_whisper import WhisperModel
from rapidfuzz.distance import Levenshtein
from rapidfuzz import process, fuzz
from mecab import MeCab
import importlib

In [8]:
# #Googledrive 마운트(Colab 사이트 사용 시 주석 해제)
# from google.colab import drive
# drive.mount('/content/drive')

#모듈파일 변경 후 세션 재시작 안 해도 되게 하는 코드
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
# 1. 파일들이 위치한 경로로 이동 colab용 
# BASE_PATH = "/content/drive/MyDrive/25-2 산학협력프로젝트/26.1_최종발표/results/"
# 프로젝트 루트로 이동 (Colab)
# %cd /content/drive/MyDrive/0208

# 모듈 Import
from config.settings import *
from utils.normalizer import TextNormalizer
from utils.data_loader import load_transcripts
from utils.metrics import calculate_cer, calculate_wer, evaluate_proper_nouns
from core.asr_engine import ASR
from core.bias_manager import BiasManager
from core.post_processor import postprocess_with_hotwords
from utils.exporter import *


In [10]:
# #그래픽카드 메모리 남용을 막기 위한 캐시 초기화

# gc.collect()
# torch.cuda.empty_cache()

# 1-5. 결과 저장 경로[현재 시간 반영해서 파일별 구분 용이]
#results 폴더 없으면 생성
import os
from config.settings import RESULTS_DIR

import random #추가
random.seed(42)

# ==============================================================================
# 메인 실행 로직
# ==============================================================================
os.makedirs(RESULTS_DIR, exist_ok=True)

now = time.gmtime(time.time()+(9*3600)) #한국 시간
formatted = time.strftime("[%Y%m%d_%H%M]", now)
OUT_ROWS = f"./results/{formatted}_asr_detail.csv"
OUT_SUM  = f"./results/{formatted}_asr_summary.csv"

# total_asr_num=len(HOTWORD_TOPK_SWEEP)

normalizer = TextNormalizer()
mecab = MeCab()
bias_mgr = BiasManager(BIAS_PATH)
transcripts = load_transcripts(TRANSCRIPTS_PATH)
files = glob.glob(os.path.join(AUDIO_FOLDER, "**/*.mp4"), recursive=True)

# ASR 모델 로드
asr = ASR(ASR_MODEL, ASR_DEVICE, ASR_COMPUTE,initial_prompt=KOREAN_ONLY_PROMPT)

rows: List[Dict[str, Any]] = []  # [수정] 결과 데이터를 저장할 리스트

# 2. 실험 루프
for top_k in HOTWORD_TOPK_SWEEP: #hotwords 개수 경우의 수 반복문
    for hotwords_strategy in HOTWORD_STRATEGY_SWEEP: #hotwords 선택 전략 경우의 수 반복문

        for bias_weight_update_cnt in BIAS_WEIGHT_UPDATE_ITERATION_SWEEP: #bias_weight_update 주기 경우의 수 반복문
            # (선택) 각 전략 시작마다 bias 초기화
            if RESET_BIASING_LIST:
                bias_mgr.reset_biasing_list(BIAS_PATH)
            # ✅ 1번 방식: 반복 횟수는 BIAS_WEIGHT_UPDATE_ITERATION_SWEEP
            for repeat in range(bias_weight_update_cnt):

                # ✅ repeat마다 hotwords 새로 샘플링 (1번 방식)
                current_hotwords = bias_mgr.get_weighted_hotwords(top_k, mode=hotwords_strategy)
                for pp_on in POSTPROCESS_SWEEP:
                    pp_str= "ON" if pp_on ==1 else "OFF"
                    print(f"\n[RUN] Top-K: {top_k} | Iteration: {repeat+1}/{bias_weight_update_cnt} | PostProcess: {pp_str}")
                    print(f"hotwords : {current_hotwords}\n")

                    #AUDIO_MAX개만큼의 파일만 뽑아 쓸 때 매번 랜덤하게 파일이 뽑히도록 랜덤 섞기
                    random.shuffle(files) # 리스트의 순서를 무작위로 섞음

                    for audio_path in files[:AUDIO_FILE_MAX]:
                        fname = os.path.basename(audio_path)
                        meta = transcripts.get(fname, {"text": "", "entities": []})

                        # 1) ASR
                        hyp_raw = asr.transcribe(audio_path, "ko", ASR_BEAM, hotwords=current_hotwords)
                        
                        # 2) 후처리
                        if pp_on:
                            hyp_final, replog = postprocess_with_hotwords(
                                hyp_raw, current_hotwords, normalizer,
                                gate=RULE_GATE, tol=RULE_TOL, wratio_th=RULE_WRATIO_TH
                            )
                        else:
                            hyp_final, replog = hyp_raw, []
                        
                        #ref_text 정규화
                        ref_final=normalizer.normalize(meta["text"], False)

                        # ✅ 3) PN 평가: 너가 수정한 4개 리턴 버전 사용
                        pn_recall, pn_cer, hyp_ents, hard_missed_ents = evaluate_proper_nouns(
                            meta.get("entities", []), hyp_final, normalizer, match_th=PN_MATCH_TH, hard_th=HARD_MISS_TH)

                        # 4) Metrics
                        # wrong_char_cnt : 틀린 음절 수, char_cnt : 전체 음절 수
                        # wrong_morph_cnt : 틀린 형태소 수, wrong_morph_cnt : 전체 형태소 수
                        cer, wrong_char_cnt, char_cnt = calculate_cer(ref_final, hyp_final, normalizer)
                        wer, wrong_morph_cnt, morph_cnt, ref_morphs, hyp_morphs = calculate_wer(ref_final, hyp_final, normalizer, mecab,meta["entities"], hyp_ents)

                        # ✅ 1번 방식: hard miss만 학습
                        bias_mgr.add_miss(hard_missed_ents)

                        #pn_recall = pn_recall if pn_recall is not None else 0.0
                        pn_recall_disp = f"{pn_recall:.4f}" if pn_recall is not None else "NA"
                        pn_cer_disp    = f"{pn_cer:.4f}"    if pn_cer    is not None else "NA"

                        # 로그
                        print(
                            f"- file: {os.path.dirname(audio_path).split('/')[-1]}/{fname} | "
                            f"pp_on={pp_on} | cer={cer:.4f} | wer={wer:.4f} | pn_cer={pn_cer_disp} | pn_recall={pn_recall_disp}" # 수정
                        )
                        print(
                            f"ref_text:  [{meta['text']}]\n"
                            f"hyp_raw:   [{hyp_raw}]\n"
                            f"hyp_final: [{hyp_final}]\n"
                            f"ref_pn:    {meta.get('entities', [])}\n"
                            f"hyp_pn:    {hyp_ents}\n"
                            f"hard_miss: {hard_missed_ents}\n"
                        )

                        # 결과 저장(컬럼명 정리 권장)
                        rows.append({
                            "file": f"{os.path.dirname(audio_path).split('/')[-1]}/{fname}",
                            "top_k": top_k,
                            "postprocess_on": int(pp_on),  # 이름 명확히
                            "hotwords_strategy": "random" if hotwords_strategy == 1 else "hybrid",
                            "bias_weight_update_cnt": repeat+1,  # 
                            "hotwords": current_hotwords,

                            "cer": f"{cer:.4f}",
                            "wer": f"{wer:.4f}",
                            "pn_recall": None if pn_recall is None else float(f"{pn_recall:.4f}"), # 수정
                            "pn_cer": None if pn_cer is None else float(f"{pn_cer:.4f}"), # 수정

                            "ref_text": meta["text"],
                            "ref_text_final" : ref_final,
                            "hyp_raw": hyp_raw,
                            "hyp_final": hyp_final,

                            "ref_text_pn": meta.get("entities", []),
                            "hyp_pn": hyp_ents,
                            "hard_missed_pn": hard_missed_ents,

                            "replog": json.dumps(replog, ensure_ascii=False),
                            "wrong_char_cnt":wrong_char_cnt,
                            "char_cnt":char_cnt,
                            "wrong_morph_cnt":wrong_morph_cnt,
                            "morph_cnt":morph_cnt,
                            "ref_morphs": ref_morphs,
                            "hyp_morphs":hyp_morphs
                        })

                # repeat 끝에 학습 반영
                bias_mgr.finalize(bias_weight_update_cnt)

save_results_with_summary(rows, OUT_ROWS)

summary = summarize(rows)
with open(OUT_SUM, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=list(summary[0].keys()))
    w.writeheader()
    w.writerows(summary)

print("\n[DONE] All experiments finished.")


[SUCCESS] /teamspace/studios/this_studio/Sanhak/lists/biasing_list.json 데이터가 모두 0으로 초기화되었습니다.

[RUN] Top-K: 20 | Iteration: 1/2 | PostProcess: ON
hotwords : ['넷플릭스', '이 사랑도 통역이 되나요', '디즈니플러스', '만달로리안', '유튜브', '침착맨', '삼국지', '티빙', '정숙한세일즈', '와이티엔', '김하성', '부부의세계', '유어아너', '나의완벽한비서', '황동혁', '투니버스', '무빙이', '웨이브', '손흥민', '바둑티브이']

{'PROTA': '프리미어리그', 'PROTB': '쿠팡플레이', 'PROTC': '손흥민'} ['PROTB', '에서', '프리미어', '리그', '경기', '중', 'PROTC', '이', '선발', '출전', '경기', '만', '하이라이트', '로', '묶', '어', '줘']
- file: Audio_files/record87.mp4 | pp_on=1 | cer=0.0000 | wer=0.1250 | pn_cer=0.0000 | pn_recall=1.0000
ref_text:  [쿠팡플레이에서 프리미어리그 경기 중 손흥민이 선발 출전 경기만 하이라이트로 묶어줘]
hyp_raw:   [쿠팡플레이 에서 프리미어 리그 경기 중 손흥민이 선발 출전 경기만 하이라이트로 묶어줘]
hyp_final: [쿠팡플레이 에서 프리미어 리그 경기 중 손흥민이 선발 출전 경기만 하이라이트로 묶어줘]
ref_pn:    ['쿠팡플레이', '프리미어리그', '손흥민']
hyp_pn:    ['쿠팡플레이', '프리미어리그', '손흥민']
hard_miss: []

{'PROTA': '케이비에스이'} ['케이비', '에스', '이', '십', '이', '에서', '이번', '주', '토요일', '밤', '에', '하', '는', '드라마', '를', '자동', '녹화', '로', '예약', '해', '줘